In [ ]:
import pandas as pd
from rhmag.data_management import FINAL_MATERIALS

In [ ]:
materials = FINAL_MATERIALS

In [ ]:
df_err = pd.read_excel('Result_Final.xlsx', sheet_name='P95Error', header=0)

err_records = []
for _, row in df_err.iterrows():
    team = row['Team']

    for mat in materials:
        seq_col = f'{mat} Seq (%)'
        ene_col = f'{mat} Ene (%)'
        if seq_col in df_err.columns and ene_col in df_err.columns:
            err_records.append({
                'model_type': team,
                'material': mat,
                'sre_95th': row[seq_col],
                'nere_95th': row[ene_col],
            })
df_err = pd.DataFrame(err_records)
df_err

In [ ]:
df_params = pd.read_excel('Result_Final.xlsx', sheet_name='Parameters', header=None)

param_records = []
for _, row in df_params.iloc[2:].iterrows():   # skip the two header rows
    team = row[0]
    for i, mat in enumerate(materials):
        col_params = 1 + i * 2   # Parameters column for this material
        param_records.append({
            'model_type': team,
            'material': mat,
            'n_params': row[col_params],
        })
df_params = pd.DataFrame(param_records)

df_params['model_type'] = df_params['model_type'].replace({
    'NITC1': 'NITC',
    'Princeton ': 'Princeton', 
})

df_params

In [ ]:
print(df_err['model_type'].unique())

In [ ]:
print(df_params['model_type'].unique())

In [ ]:
df_external = pd.merge(df_err, df_params, on=['model_type', 'material'])

In [ ]:
df_external

In [ ]:
df_external[['sre_95th', 'nere_95th']] = df_external[['sre_95th', 'nere_95th']] / 100

In [ ]:
df_external

In [ ]:
# save to parquet
df_external.to_parquet("external_pareto_results.parquet")